# Cleaning Data — Retail Transactions
**Oasis Infobyte SIP — Data Analytics, Level 1, Task 3**

**Objective:** take a deliberately messy dataset and turn it into a clean, analysis-ready one, documenting every decision.

> Dataset note: `messy_retail_data.csv` is a sample retail dataset with deliberately injected problems (missing values, duplicates, inconsistent text, mixed date formats, text-formatted prices and outliers).

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv('messy_retail_data.csv')
df = raw.copy()
raw.head(10)

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Product,Quantity,Price per Unit,Total Amount
0,T00940,2023-03-06,C1085,MALE,36.0,Clothing,Jacket,2,5585.94,11171.88
1,T02817,2023-07-16,C1258,FEMALE,36.0,Electronics,Smartwatch,1,8439.59,8439.59
2,T01911,2023-07-02,C1171,Female,47.0,Electronics,Power Bank,1,2413.2,2413.20
3,T00667,2024-09-11,C1059,Female,18.0,Beauty,Lipstick,1,334.44,334.44
4,T02188,2023-02-10,C1200,Male,33.0,Electronics,Smartwatch,1,7717.37,7717.37
5,T01241,2023-10-24,C1111,Female,47.0,NaN,Jacket,1,5756.68,5756.68
6,T02664,2023-11-03,C1239,FEMALE,41.0,Beauty,Lipstick,1,1678.37,1678.37
7,T04340,24/02/2023,C1399,Female,32.0,Electronics,Headphones,1,5496.85,5496.85
8,T02032,2023-09-04,C1187,m,63.0,BEAUTY,Face Wash,2,565.44,1130.88
9,T03515,2023-06-15,C1313,Female,37.0,Books,Novel,1,351.42,351.42


## 1. Data quality report (before cleaning)

In [2]:
def quality_report(d):
    return {
        'rows': len(d),
        'duplicate_rows': int(d.duplicated().sum()),
        'total_nulls': int(d.isnull().sum().sum()),
    }

before = quality_report(raw)
print(before)
print('\nNulls per column:\n', raw.isnull().sum())
print('\nData types:\n', raw.dtypes)
print('\nUnique Gender values:', raw['Gender'].unique())
print('Unique Category values:', raw['Product Category'].unique())
print('\nAge range:', raw['Age'].min(), 'to', raw['Age'].max())

{'rows': 2580, 'duplicate_rows': 80, 'total_nulls': 367}

Nulls per column:
 Transaction ID        0
Date                  0
Customer ID           0
Gender                0
Age                 152
Product Category    124
Product               0
Quantity              0
Price per Unit       91
Total Amount          0
dtype: int64

Data types:
 Transaction ID          str
Date                    str
Customer ID             str
Gender                  str
Age                 float64
Product Category        str
Product                 str
Quantity              int64
Price per Unit          str
Total Amount        float64
dtype: object

Unique Gender values: <StringArray>
['MALE', 'FEMALE', 'Female', 'Male', 'm', 'F', 'female', 'f', 'male', 'M']
Length: 10, dtype: str
Unique Category values: <StringArray>
[       'Clothing',     'Electronics',    'Electronics ',         'Beauty ',
               nan,          'Beauty',          'BEAUTY',           'Books',
 'Home & Kitchen ',  'HOME & KITCHE

**Problems found:**
- Duplicate rows
- Missing values in `Age`, `Product Category` and `Price per Unit`
- `Gender` written many ways (male / M / MALE / f / ...)
- `Product Category` has inconsistent case and trailing spaces
- `Date` stored as text in two different formats
- `Price per Unit` is text because some values look like `Rs.1234.50`
- `Age` contains impossible values (negative, 150, 999)
- A few extreme prices (outliers)

## 2. Remove duplicate rows

In [3]:
n_before = len(df)
df = df.drop_duplicates()
print('Duplicate rows removed:', n_before - len(df))

Duplicate rows removed: 80


## 3. Fix data types and standardise formats
- Prices: strip the `Rs.` prefix and convert to float.
- Dates: two formats are present (`YYYY-MM-DD` and `DD/MM/YYYY`), so each is parsed separately.
- IDs stay as strings.

In [4]:
df['Price per Unit'] = pd.to_numeric(df['Price per Unit'].astype(str).str.replace('Rs.', '', regex=False), errors='coerce')

iso = pd.to_datetime(df['Date'], format='%Y-%m-%d', errors='coerce')
dmy = pd.to_datetime(df['Date'], format='%d/%m/%Y', errors='coerce')
df['Date'] = iso.fillna(dmy)

df['Customer ID'] = df['Customer ID'].astype(str)
df['Transaction ID'] = df['Transaction ID'].astype(str)
print('Unparsed dates:', df['Date'].isnull().sum())
df.dtypes

Unparsed dates: 0


Transaction ID                 str
Date                datetime64[us]
Customer ID                    str
Gender                         str
Age                        float64
Product Category               str
Product                        str
Quantity                     int64
Price per Unit             float64
Total Amount               float64
dtype: object

In [5]:
# Gender: map every variant to 'Male' / 'Female'
g = df['Gender'].str.strip().str.lower()
df['Gender'] = g.map({'male':'Male','m':'Male','female':'Female','f':'Female'})

# Category: trim spaces and use consistent title case
df['Product Category'] = df['Product Category'].str.strip().str.title()
print(df['Gender'].value_counts(dropna=False))
print(df['Product Category'].value_counts(dropna=False))

Gender
Male      1276
Female    1224
Name: count, dtype: int64
Product Category
Clothing          573
Electronics       571
Beauty            442
Home & Kitchen    412
Books             382
NaN               120
Name: count, dtype: int64


## 4. Handle impossible values and outliers
**Age:** valid customer ages are 18–100, so values outside that range are data-entry errors. They are set to missing and then imputed with the median (see step 5).

**Price per Unit:** outliers are detected with the **IQR method**, calculated separately for each product.

In [6]:
df.loc[(df['Age'] < 18) | (df['Age'] > 100), 'Age'] = np.nan

# IQR method applied per product (an Electronics price is not an outlier just because it is higher than a Book price)
def iqr_bounds(x):
    q1, q3 = x.quantile(0.25), x.quantile(0.75)
    i = q3 - q1
    return pd.Series({'low': q1 - 1.5*i, 'high': q3 + 1.5*i})

bounds = df.groupby('Product')['Price per Unit'].apply(iqr_bounds).unstack()
df = df.join(bounds, on='Product')
is_outlier = (df['Price per Unit'] < df['low']) | (df['Price per Unit'] > df['high'])
print('Price outliers found:', is_outlier.sum())
df.loc[is_outlier, ['Transaction ID','Product','Price per Unit','low','high']].head()

Price outliers found: 15


,Transaction ID,Product,Price per Unit,low,high
81,T02087,Self-Help Book,150095.52,-84.29000,1299.87000
282,T01334,Bedsheet,150095.52,-97.02000,3515.40000
305,T00922,Jacket,150095.52,-833.64875,9368.44125
549,T03851,Comic,150095.52,-16.53500,717.50500
835,T04215,Moisturizer,150095.52,-119.10500,1641.01500


**Decision:** the flagged prices are far outside the normal range of the same product (several times higher), so they are entry errors, not genuine sales. They are set to missing and re-imputed with the median price of the same product. This keeps the row while removing the distortion.

In [7]:
df.loc[is_outlier, 'Price per Unit'] = np.nan
df = df.drop(columns=['low','high'])

## 5. Handle missing values
| Column | Strategy | Reason |
|---|---|---|
| `Age` | Median | Numeric and slightly skewed; the median is robust to outliers |
| `Price per Unit` | Median of the same product | Prices differ a lot between products, so a global mean would be misleading |
| `Product Category` | Mode | Categorical column; a missing category is filled with the most common one |
| `Total Amount` | Recalculated as `Quantity × Price per Unit` | It is derived, so it is better to recompute than to impute |

In [8]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Product Category'] = df['Product Category'].fillna(df['Product Category'].mode()[0])
df['Price per Unit'] = df['Price per Unit'].fillna(df.groupby('Product')['Price per Unit'].transform('median'))
df['Total Amount'] = (df['Quantity'] * df['Price per Unit']).round(2)
df['Age'] = df['Age'].astype(int)
print(df.isnull().sum())

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Product             0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64


## 6. Before vs. after summary

In [9]:
after = quality_report(df)
def type_accuracy(d):
    from pandas.api import types as t
    checks = [t.is_datetime64_any_dtype(d['Date']), t.is_integer_dtype(d['Age']), t.is_float_dtype(d['Price per Unit']),
              t.is_integer_dtype(d['Quantity']), t.is_float_dtype(d['Total Amount'])]
    return f'{sum(checks)}/5 correct'

summary = pd.DataFrame({
    'Before cleaning': [before['rows'], before['duplicate_rows'], before['total_nulls'], type_accuracy(raw)],
    'After cleaning':  [after['rows'],  after['duplicate_rows'],  after['total_nulls'],  type_accuracy(df)],
}, index=['Row count','Duplicate rows','Total null values','Columns with correct dtype'])
summary

,Before cleaning,After cleaning
Row count,2580,2500
Duplicate rows,80,0
Total null values,367,0
Columns with correct dtype,2/5 correct,5/5 correct


## 7. Save the cleaned dataset

In [10]:
df = df.sort_values('Date').reset_index(drop=True)
df.to_csv('cleaned_retail_data.csv', index=False)
print('Saved cleaned_retail_data.csv with', len(df), 'rows')
df.head()

Saved cleaned_retail_data.csv with 2500 rows


,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Product,Quantity,Price per Unit,Total Amount
0,T02574,2023-01-01,C1231,Female,27,Electronics,Smartwatch,1,12196.33,12196.33
1,T00123,2023-01-01,C1010,Female,35,Electronics,Headphones,4,2194.27,8777.08
2,T02780,2023-01-01,C1256,Female,18,Electronics,Bluetooth Speaker,1,4482.08,4482.08
3,T00858,2023-01-01,C1077,Female,46,Books,Novel,1,611.53,611.53
4,T03757,2023-01-01,C1338,Male,52,Electronics,Bluetooth Speaker,2,5960.06,11920.12


## Conclusion
Starting from a messy file, the data was deduplicated, standardised (dates, gender, categories, prices), corrected for impossible ages and extreme prices, and every missing value was filled with a justified strategy. The result is saved as `cleaned_retail_data.csv`, ready for analysis.